# Experiment 7: time curves of a cold front, reality vs forecast

**Time curves** (Bach et al. 2015): every dot is one snapshot, here **a map of Austria at one hour**. Dots are
placed so that similar maps are close together and different maps far apart (classical MDS on the distances
between all pairs of maps). A line connects the dots in time order. The axes have no meaning, only closeness does.

**Window:** 5 days around the strongest front (25-26 March 2026).
**Maps:** temperature **anomalies** (minus the usual value of that point at that month and hour), so the normal
day-night swing does not dominate. Distance between two maps = root of the summed squared differences.

All 378 points (a first test used 100 points and one map every 2 h).
- Step 1: reality only, one map per hour (no model).
- Step 2: reality and forecast, one map per hour. Forecast map for hour t = the forecast made **6 h earlier**
  (temperature only, 10 weeks of history, median).

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from tirex2 import TimeseriesType, load_model

DATA = Path("/workspace/data/austria_15km")
FIG = Path("/workspace/phase1-model-behavior/pair-1/experiments/figures/exp7")
FIG.mkdir(parents=True, exist_ok=True)
C = 10 * 168
LEAD = 6          # forecast map for hour t comes from the run started at t - LEAD
N_POINTS = 378    # all points (the first test used 100)
STEP = 1          # hours between forecast snapshots (the first test used 2)
RERUN = False
QUICK = bool(os.environ.get("QUICK"))
if QUICK:
    N_POINTS, STEP, FIG = 20, 12, Path("/tmp")
RESULTS = Path(f"/workspace/data/notebooks/results/exp7_time_curves_{N_POINTS}pts_{STEP}h.npz")
if QUICK:
    RESULTS = Path("/tmp/exp7_test.npz")

In [2]:
df = pd.read_parquet(DATA / "grid_long.parquet", columns=["point_id", "time", "T2M"])
Tw = df.pivot(index="time", columns="point_id", values="T2M").interpolate(limit=3, limit_area="inside")
times, T = Tw.index, Tw.to_numpy(np.float32)
usual = pd.DataFrame(T).groupby([times.month, times.hour]).transform("mean").to_numpy(np.float32)
A = T - usual

fronts = pd.read_csv(DATA / "fronts.csv", parse_dates=["peak_time"])
f = fronts[fronts.cold_front].sort_values("share", ascending=False)
ev = pd.read_parquet(DATA / "events.parquet")
ev = ev[ev.kind == "cold_front"]
peak = pd.Timestamp("2026-03-25 22:00", tz=times.tz)
in_front = ev[(ev.time >= peak - pd.Timedelta("1D")) & (ev.time <= peak + pd.Timedelta("1D"))].groupby("time").point_id.nunique()
centre = in_front.idxmax()  # hour with most points in the passage
win = np.arange(times.get_loc(centre) - 60, times.get_loc(centre) + 60)
pcols = np.linspace(0, T.shape[1] - 1, N_POINTS).astype(int)
share = (in_front.reindex(times[win]).fillna(0) / T.shape[1]).to_numpy()
print(f"window {times[win[0]]:%a %d %b %H:%M} to {times[win[-1]]:%a %d %b %H:%M} UTC, centre {centre:%a %d %b %H:%M}; {N_POINTS} points")


def mds(X):
    """Classical MDS: 2D positions whose distances best match the distances between the rows of X."""
    sq = (X ** 2).sum(1)
    D2 = np.maximum(sq[:, None] + sq[None] - 2 * X @ X.T, 0)
    n = len(X)
    J = np.eye(n) - 1 / n
    w, v = np.linalg.eigh(-0.5 * J @ D2 @ J)
    i = np.argsort(w)[::-1][:2]
    xy = v[:, i] * np.sqrt(np.maximum(w[i], 0))
    kept = w[i].sum() / w[w > 0].sum()
    return xy, kept


def label(t):
    return f"{t:%a %d %b %H:%M}"


def ticks(tt):
    """Indices of the 00:00 and 12:00 hours (labelled on the curve)."""
    return [k for k, t in enumerate(tt) if t.hour in (0, 12)]

window Mon 23 Mar 21:00 to Sat 28 Mar 20:00 UTC, centre Thu 26 Mar 09:00; 378 points


## Step 1: reality only (one map per hour, no model)

**How to read it:** each dot is **one hour**, and stands for the **whole temperature map of Austria** at that hour
(all points), as the difference from the **monthly usual temperature** of that place at that hour (its average
over all days of that month at that hour). Dots close together =
similar maps; far apart = very different maps. The axes have no meaning. The line connects the hours in time order.

Colours: the same red-blue scale as the maps. A dot's colour = the average of its map (red = Austria warmer than
the monthly usual, blue = colder). White ring = **front passing** (at least 10 % of all points in a labelled
cold-front passage). A, B, C = three example hours; their maps are shown on the right.

In [3]:
R1 = A[win][:, pcols]
xy1, kept1 = mds(R1)
tt = times[win]
P = pd.read_csv(DATA / "points_with_labels.csv").set_index("point_id").loc[Tw.columns[pcols]]
ip = np.where(share >= 0.1)[0]
phase = np.where(np.arange(len(win)) < ip[0], 0, np.where(np.arange(len(win)) <= ip[-1], 1, 2))
PH = [("before the front", "#f4a259"), ("front passing", "#ff4d6d"), ("after the front", "#4cc9f0")]
before = np.where(phase == 0)[0]
keys = {"A": int(before[np.argmax(R1[before].mean(1))]),      # warmest map before the front
        "B": int(np.argmax(share)),                           # most points in the passage
        "C": int(min(ip[-1] + 24, len(win) - 1))}             # one day after the passage
key_names = {"A": "before", "B": "front passing", "C": "a day after"}

fig = make_subplots(rows=3, cols=2, column_widths=[0.72, 0.28], horizontal_spacing=0.02, vertical_spacing=0.06,
                    specs=[[{"rowspan": 3}, {"type": "geo"}], [None, {"type": "geo"}], [None, {"type": "geo"}]],
                    subplot_titles=[""] + [f"{k}: {key_names[k]} ({tt[i]:%a %H}h)" for k, i in keys.items()])
fig.add_trace(go.Scatter(x=xy1[:, 0], y=xy1[:, 1], mode="lines", line={"color": "#777777", "width": 1.5, "shape": "spline"},
                         hoverinfo="skip", showlegend=False), row=1, col=1)
CLIM = 8  # same red-blue scale for the curve dots and the maps (°C vs monthly usual)
fig.add_trace(go.Scatter(x=xy1[:, 0], y=xy1[:, 1], mode="markers", showlegend=False,
                         marker={"color": R1.mean(1), "colorscale": "RdBu_r", "cmin": -CLIM, "cmax": CLIM, "size": 9,
                                     "line": {"color": np.where(phase == 1, "white", "rgba(0,0,0,0)"), "width": 2}},
                         text=[f"{label(t)}<br>Austria on average {a:+.1f} °C vs monthly usual<br>{100 * s:.0f} % of points in the front passage"
                               for t, a, s in zip(tt, R1.mean(1), share)],
                         hovertemplate="%{text}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="front passing (white ring)",
                         marker={"color": "rgba(0,0,0,0)", "size": 9, "line": {"color": "white", "width": 2}}), row=1, col=1)
# label offsets in pixels (right, down), placed by hand so they sit beside the dots; None = no label
LABEL_POS = {"Tue 00h": (22, 30), "Tue 12h": (35, 25), "Wed 00h": (45, 25), "Wed 12h": (40, 0), "Thu 00h": (0, 45),
             "Thu 12h": (30, -20), "Fri 00h": None, "Fri 12h": (-25, 20), "Sat 00h": (0, 30), "Sat 12h": (35, -15)}
for i in ticks(tt):
    lab = f"{tt[i]:%a %H}h"
    pos = LABEL_POS.get(lab, (40, 0))
    if pos is None:
        continue
    fig.add_annotation(x=xy1[i, 0], y=xy1[i, 1], text=lab, showarrow=False, xshift=pos[0], yshift=-pos[1], xref="x", yref="y",
                       font={"size": 12, "color": "white"}, bgcolor="rgba(17,17,17,0.8)", borderpad=2)
KEY_POS = {"A": (-36, 0), "B": (30, -26), "C": (-42, 16)}  # box position vs dot (right, down) in pixels
for k, i in keys.items():
    fig.add_annotation(x=xy1[i, 0], y=xy1[i, 1], text=f"<b>{k}</b>", showarrow=True, arrowcolor="white", arrowwidth=1.2,
                       ax=KEY_POS[k][0], ay=KEY_POS[k][1], xref="x", yref="y", font={"size": 16, "color": "white"},
                       bgcolor="rgba(0,0,0,0)", borderwidth=0, borderpad=3)
for r, (k, i) in enumerate(keys.items(), start=1):
    fig.add_trace(go.Scattergeo(lon=P.lon, lat=P.lat, mode="markers", showlegend=False,
                                marker={"color": R1[i], "colorscale": "RdBu_r", "cmin": -CLIM, "cmax": CLIM, "size": 5,
                                            "colorbar": {"title": "°C vs monthly<br>usual", "x": 1.0, "y": 0.5, "len": 0.9} if r == 1 else None,
                                            "showscale": r == 1},
                                hovertemplate="%{marker.color:+.1f} °C vs monthly usual<extra></extra>"), row=r, col=2)
fig.update_geos(fitbounds="locations", showcountries=True, countrycolor="#666", showland=True, landcolor="#1e1e1e",
                bgcolor="rgba(0,0,0,0)", resolution=50)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, scaleanchor="x", row=1, col=1)  # same scale on both axes, so distances are not stretched
fig.update_layout(height=720, template="plotly_dark", legend={"orientation": "h", "y": -0.03, "x": 0},
                  title="How Austria's temperature map changes over 5 days around the 25-26 March cold front"
                        "<br><sup>each dot = one hour = the whole map of Austria (difference from the monthly usual temperature at that hour); "
                        f"close dots = similar maps, axes have no meaning; 2D layout keeps {kept1:.0%} of the differences</sup>",
                  margin={"t": 100, "r": 60})
EDIT = {"edits": {"annotationPosition": True, "annotationTail": True}}  # labels can still be dragged with the mouse (not saved)
fig.write_html(FIG / "exp7_reality.html", include_plotlyjs="cdn", config=EDIT)
fig.show(config=EDIT)

## Step 2: reality vs forecast (one map per hour)

Each dot is the **forecast** map of Austria for one hour. How it is made: for every hour t and every point, a
separate run starts at t − 6 h (the model knows everything up to t − 7 h) and forecasts 24 h; only its value for
hour t (the 7th forecast hour, so **7 h ahead**) is used here; the other 23 values are saved, not used in this figure. So every dot has the same head start (lead-time mode), 120 hours × all points = one run per point
and hour. Forecast and real maps were laid out **together**, so the faint line (the real path of Step 1) is on the
same layout. Colours and rings as in Step 1; A, B, C mark the real maps of Step 1, A′, B′, C′ the forecasts for
the same hours.

In [4]:
sel = win[::STEP]
model = load_model("NX-AI/TiRex-2", device="cpu")
if RESULTS.exists() and not RERUN and not QUICK:
    z = np.load(RESULTS)
    q = z["forecast"]
    print(f"loaded saved forecasts from {RESULTS.name}")
else:
    t0 = time.time()
    ts = [TimeseriesType(torch.from_numpy(T[t - LEAD - C : t - LEAD, c][None].copy()), None, None) for t in sel for c in pcols]
    out = model.forecast(ts, prediction_length=24, output_type="numpy", batch_size=128)
    q_all = np.stack([o[0][[0, 4, 8]] for o in out]).reshape(len(sel), len(pcols), 3, 24)  # runs started at t - LEAD, all 24 leads
    q = q_all[:, :, :, LEAD]  # (hours, points, q10/q50/q90) for hour t
    print(f"{len(ts)} forecasts in {time.time() - t0:.0f} s")
    np.savez_compressed(RESULTS, times=np.asarray(times[sel].astype(str)).astype("U"), point_id=Tw.columns[pcols].to_numpy(),
                        truth=T[sel][:, pcols], usual=usual[sel][:, pcols], forecast=q, lead=LEAD,
                        forecast_all_leads=q_all.astype(np.float16))
R2 = A[sel][:, pcols]
F2 = q[:, :, 1] - usual[sel][:, pcols]
xy, kept = mds(np.vstack([R2, F2]))
xr, xf = xy[: len(sel)], xy[len(sel):]
ts2 = times[sel]
err = np.sqrt(((F2 - R2) ** 2).mean(1))  # map error per hour (°C, root mean square over the points)
sh2 = share[::STEP]

ph2 = phase[::STEP]
keys2 = {k: i // STEP for k, i in keys.items()}
# only the forecast path (the real path is shown above in Step 1); same layout as computed together with reality
fig = go.Figure()
fig.add_trace(go.Scatter(x=xr[:, 0], y=xr[:, 1], mode="lines", line={"color": "rgba(200,200,200,0.3)", "width": 1.2, "shape": "spline"},
                         name="real path (for reference)", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=xf[:, 0], y=xf[:, 1], mode="lines", line={"color": "#777777", "width": 1.5, "shape": "spline"},
                         hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=xf[:, 0], y=xf[:, 1], mode="markers", showlegend=False,
                         marker={"color": F2.mean(1), "colorscale": "RdBu_r", "cmin": -CLIM, "cmax": CLIM, "size": 9,
                                     "line": {"color": np.where(ph2 == 1, "white", "rgba(0,0,0,0)"), "width": 2},
                                     "colorbar": {"title": "°C vs<br>monthly<br>usual"}},
                         text=[f"{label(t)} (forecast)<br>Austria on average {a:+.1f} °C vs monthly usual<br>forecast map error {e:.1f} °C"
                               for t, a, e in zip(ts2, F2.mean(1), err)],
                         hovertemplate="%{text}<extra></extra>"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="front passing (white ring)",
                         marker={"color": "rgba(0,0,0,0)", "size": 9, "line": {"color": "white", "width": 2}}))
for k, i in keys2.items():  # A, B, C on the real path (grey, like the reference line); A', B', C' on the forecast path
    fig.add_annotation(x=xr[i, 0], y=xr[i, 1], text=f"<b>{k}</b>", showarrow=True, arrowcolor="#aaaaaa", arrowwidth=1.2,
                       ax=-KEY_POS[k][0], ay=-KEY_POS[k][1], font={"size": 15, "color": "#aaaaaa"}, bgcolor="rgba(0,0,0,0)", borderwidth=0)
    fig.add_annotation(x=xf[i, 0], y=xf[i, 1], text=f"<b>{k}′</b>", showarrow=True, arrowcolor="white", arrowwidth=1.2,
                       ax=KEY_POS[k][0], ay=KEY_POS[k][1], font={"size": 16, "color": "white"}, bgcolor="rgba(0,0,0,0)", borderwidth=0)
pad = 0.05 * np.ptp(xy, axis=0)
fig.update_xaxes(visible=False, range=[xy[:, 0].min() - pad[0], xy[:, 0].max() + pad[0]])
fig.update_yaxes(visible=False, range=[xy[:, 1].min() - pad[1], xy[:, 1].max() + pad[1]], scaleanchor="x")
fig.update_layout(height=560, template="plotly_dark", legend={"orientation": "h", "y": -0.03, "x": 0},
                  title=f"Does the forecast follow the front? Forecast maps ({LEAD + 1} h ahead) on the same layout as reality"
                        "<br><sup>each dot = the forecast map of Austria for one hour, made 7 h after the last known hour; faint line = the real path from Step 1; "
                        "A, B, C = real maps of Step 1, A′, B′, C′ = forecasts for the same hours</sup>", margin={"t": 100})
fig.write_html(FIG / "exp7_reality_vs_forecast.html", include_plotlyjs="cdn", config=EDIT)
fig.show(config=EDIT)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

loaded saved forecasts from exp7_time_curves_378pts_1h.npz


## Reusing the results

Saved in `data/notebooks/results/exp7_time_curves_378pts_1h.npz` (the 100-point test in `exp7_time_curves_100pts_2h.npz`):

```python
z = np.load("/workspace/data/notebooks/results/exp7_time_curves_378pts_1h.npz")
z["times"], z["point_id"]      # the 120 hours of the window (UTC) and the 378 points
z["truth"], z["usual"]         # (hours, points) real temperature and the usual value for that month and hour, °C
z["forecast"]                  # (hours, points, q10/q50/q90) forecast for that hour, made 6 h earlier
z["forecast_all_leads"]        # (hours, points, q10/q50/q90, 24 leads) the whole 24 h of each run started 6 h before each hour
```

### Check without the 2D layout: map error over time

The same comparison on a normal time axis, to check what the time curve suggests (the 2D layout distorts
distances a little).

In [5]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=ts2, y=100 * sh2, name="points in front passage (%)", marker_color="rgba(31,119,180,0.25)"), secondary_y=True)
fig.add_trace(go.Scatter(x=ts2, y=err, mode="lines+markers", name="map error (°C)", line={"color": "#ff7f0e", "width": 2.5}), secondary_y=False)
fig.update_yaxes(title_text="forecast map error (°C)", rangemode="tozero", secondary_y=False)
fig.update_yaxes(title_text="points in front passage (%)", range=[0, 100], secondary_y=True, showgrid=False)
fig.update_layout(height=380, template="plotly_dark", legend={"orientation": "h", "y": -0.2},
                  title=f"How wrong is the forecast map ({LEAD} h ahead) over the 5 days?")
fig.write_html(FIG / "exp7_error_over_time.html", include_plotlyjs="cdn")
fig.show()
print(f"map error: mean {err.mean():.2f} °C, max {err.max():.2f} °C at {label(ts2[err.argmax()])}")

map error: mean 1.96 °C, max 7.49 °C at Thu 26 Mar 12:00
